In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, mean_absolute_error
from sklearn.impute import SimpleImputer

In [ ]:
# Загрузка данных
data = pd.read_csv('D:/my_ML/diploma_polytech/data/raw/vehicle_ins_data_1.csv', sep = ";",index_col= False)

# Создание целевой переменной: 1 если N_claims_year > 1, иначе 0
data['claim_prob'] = (data['N_claims_year'] > 1).astype(int)

# Предобработка данных
# Удаление ненужных столбцов
cols_to_drop = ['ID', 'Date_start_contract', 'Date_last_renewal', 'Date_next_renewal', 
                'Date_lapse', 'N_claims_year', 'Cost_claims_year', 'N_claims_history', 
                'R_Claims_history']
data = data.drop(columns=cols_to_drop)

C:\Users\andre\AppData\Local\Temp\ipykernel_19548\1202950708.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('D:/my_ML/diploma_polytech/data/raw/vehicle_ins_data_1.csv', sep = ";",index_col= False)


In [4]:
# 1. Сначала обработать ВСЕ датовые колонки, если они существуют
date_columns = ['Date_birth', 'Date_driving_licence']

# Создаем список существующих датовых колонок
existing_date_columns = [col for col in date_columns if col in data.columns]

# Проверим, какие датовые колонки нашлись
print("Существующие датовые колонки для обработки:", existing_date_columns)

if not existing_date_columns:
    print("Предупреждение: Датовые колонки не найдены в DataFrame.")
else:
    for col in existing_date_columns:
        data[col] = pd.to_datetime(data[col], format='%d/%m/%Y', errors='coerce')

    # 2. Извлечь числовые признаки из существующих дат
    reference_date = pd.to_datetime('2019-12-31')
    
    if 'Date_birth' in data.columns:
        data['Age'] = (reference_date - data['Date_birth']).dt.days // 365
    
    if 'Date_driving_licence' in data.columns:
        data['Driving_experience'] = (reference_date - data['Date_driving_licence']).dt.days // 365

    

# 2. Извлечь числовые признаки из дат
reference_date = pd.to_datetime('2019-12-31')
data['Age'] = (reference_date - data['Date_birth']).dt.days // 365
data['Driving_experience'] = (reference_date - data['Date_driving_licence']).dt.days // 365

# 3. Удалить исходные датовые колонки
data = data.drop(columns=date_columns)
data = data.drop(columns=existing_date_columns, errors='ignore')
# 4. Обработать категориальные переменные
categorical_cols = ['Type_fuel']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# 5. Убедиться, что все данные числовые
data = data.apply(pd.to_numeric, errors='coerce')

# 6. Обработка пропущенных значений
imputer = SimpleImputer(strategy='median')
data_imputed = imputer.fit_transform(data)
data = pd.DataFrame(data_imputed, columns=data.columns)



Существующие датовые колонки для обработки: ['Date_birth', 'Date_driving_licence']


In [5]:
# Разделение на признаки и целевую переменную
X = data.drop(columns=['claim_prob'])
y = data['claim_prob']

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Масштабирование признаков
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Преобразование в тензоры PyTorch
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)



In [6]:
# Создание нейронной сети
class InsuranceNet(nn.Module):
    def __init__(self, input_size):
        super(InsuranceNet, self).__init__()
        self.layer1 = nn.Linear(input_size, 64)
        self.layer2 = nn.Linear(64, 48)
        self.layer3 = nn.Linear(48, 24)
        self.layer4 = nn.Linear(24, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.relu(self.layer3(x))
        x = self.sigmoid(self.layer4(x))
        return x

# Инициализация модели
input_size = X_train_tensor.shape[1]
model = InsuranceNet(input_size)

# Определение функции потерь и оптимизатора
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)



In [7]:
# Обучение модели
epochs = 300
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')


Epoch [10/300], Loss: 0.6444
Epoch [20/300], Loss: 0.5422
Epoch [30/300], Loss: 0.4092
Epoch [40/300], Loss: 0.3261
Epoch [50/300], Loss: 0.3182
Epoch [60/300], Loss: 0.3060
Epoch [70/300], Loss: 0.3006
Epoch [80/300], Loss: 0.2982
Epoch [90/300], Loss: 0.2963
Epoch [100/300], Loss: 0.2953
Epoch [110/300], Loss: 0.2943
Epoch [120/300], Loss: 0.2936
Epoch [130/300], Loss: 0.2930
Epoch [140/300], Loss: 0.2924
Epoch [150/300], Loss: 0.2919
Epoch [160/300], Loss: 0.2915
Epoch [170/300], Loss: 0.2910
Epoch [180/300], Loss: 0.2906
Epoch [190/300], Loss: 0.2902
Epoch [200/300], Loss: 0.2899
Epoch [210/300], Loss: 0.2895
Epoch [220/300], Loss: 0.2892
Epoch [230/300], Loss: 0.2888
Epoch [240/300], Loss: 0.2885
Epoch [250/300], Loss: 0.2882
Epoch [260/300], Loss: 0.2879
Epoch [270/300], Loss: 0.2876
Epoch [280/300], Loss: 0.2873
Epoch [290/300], Loss: 0.2870
Epoch [300/300], Loss: 0.2867


In [8]:
# Оценка модели
model.eval()
with torch.no_grad():
    y_pred_proba = model(X_test_tensor).numpy()
    y_pred_binary = (y_pred_proba > 0.5).astype(int)

# Метрики
roc_auc = roc_auc_score(y_test, y_pred_proba)
mae = mean_absolute_error(y_test, y_pred_proba)

print(f'ROC-AUC: {roc_auc:.4f}')
print(f'MAE: {mae:.4f}')

with torch.no_grad():
    train_proba = torch.sigmoid(model(X_train_tensor)).numpy().flatten()
train_auc = roc_auc_score(y_train, train_proba)
print(f"\nROC-AUC на тренировочных данных: {train_auc:.4f}")

ROC-AUC: 0.7237
MAE: 0.1624

ROC-AUC на тренировочных данных: 0.7221
